In [0]:
from pyspark.sql import functions as F

bronze = spark.table("nyc_taxi.bronze.yellow_tripdata")
 
bronze.select("trip_distance", "fare_amount", "total_amount", "passenger_count").summary().show()
 
# How extreme is the tail?
bronze.select(F.max("trip_distance"), F.max("fare_amount")).show()
 
# Are there trips with impossible durations?
dur = bronze.withColumn("duration_min",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60)
dur.select("duration_min").summary("min", "25%", "50%", "75%", "max").show()


In [0]:
bronze = spark.table("nyc_taxi.bronze.yellow_tripdata")
bronze.approxQuantile("trip_distance", [0.99, 0.999, 0.9999, 0.99999], 0.0001)

In [0]:
from pyspark.sql import functions as F

DISTANCE_CEILING = 100  # 99.9th percentile was 29.79mi; next percentile jumps to 398,608mi (data error) — 100 sits cleanly between real and broken

bronze = bronze.withColumn("_payment_type_safe", F.coalesce(F.col("payment_type"), F.lit(-1)))

tagged = bronze.withColumn("_reject_reason",
    F.when(F.col("tpep_pickup_datetime").isNull() | F.col("tpep_dropoff_datetime").isNull(),
           "missing_timestamp")
     .when(F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime"),
           "dropoff_before_pickup")
     .when(F.col("trip_distance") <= 0,
           "zero_or_negative_distance")
     .when(F.col("trip_distance") > DISTANCE_CEILING,
           "implausible_distance")
     .when(F.col("passenger_count").isNull(),
           "null_passenger_count")
     .when(F.col("passenger_count") == 0,
           "zero_passenger_count")
     .when(F.col("passenger_count") > 6,
           "implausible_passenger_count")
     .when((F.col("fare_amount") < 3.00) & (~F.col("_payment_type_safe").isin(3, 4)),
           "fare_below_minimum")
     .when((F.col("total_amount") < 4.50) & (~F.col("_payment_type_safe").isin(3, 4)),
           "total_below_minimum")
     .otherwise(None)
).drop("_payment_type_safe")

clean      = tagged.filter(F.col("_reject_reason").isNull()).drop("_reject_reason")
quarantine = tagged.filter(F.col("_reject_reason").isNotNull())

print("clean:     ", clean.count())
print("quarantine:", quarantine.count())
quarantine.groupBy("_reject_reason").count().orderBy(F.desc("count")).show()

In [0]:
from pyspark.sql import functions as F

bronze = spark.table("nyc_taxi.bronze.yellow_tripdata")

DISTANCE_CEILING = 100  # 99.9th percentile was 29.79mi; next jump was 398,608mi (two isolated errors) — 100 sits cleanly between real and broken

bronze = bronze.withColumn("_payment_type_safe", F.coalesce(F.col("payment_type"), F.lit(-1)))

tagged = bronze.withColumn("_reject_reason",
    F.when(F.col("tpep_pickup_datetime").isNull() | F.col("tpep_dropoff_datetime").isNull(),
           "missing_timestamp")
     .when(F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime"),
           "dropoff_before_pickup")
     .when(F.col("trip_distance") <= 0,
           "zero_or_negative_distance")
     .when(F.col("trip_distance") > DISTANCE_CEILING,
           "implausible_distance")
     .when(F.col("passenger_count").isNull(),
           "null_passenger_count")
     .when(F.col("passenger_count") == 0,
           "zero_passenger_count")
     .when(F.col("passenger_count") > 6,
           "implausible_passenger_count")
     .when(F.col("fare_amount") < 0,
           "negative_fare")
     .when((F.col("fare_amount") < 3.00) & (~F.col("_payment_type_safe").isin(3, 4)),
           "fare_below_minimum")
     .when(F.col("total_amount") < 0,
           "negative_total")
     .when((F.col("total_amount") < 4.50) & (~F.col("_payment_type_safe").isin(3, 4)),
           "total_below_minimum")
     .otherwise(None)
).drop("_payment_type_safe")

clean      = tagged.filter(F.col("_reject_reason").isNull()).drop("_reject_reason")
quarantine = tagged.filter(F.col("_reject_reason").isNotNull())

print("clean:     ", clean.count())
print("quarantine:", quarantine.count())
quarantine.groupBy("_reject_reason").count().orderBy(F.desc("count")).show()

In [0]:
clean.filter((F.col("fare_amount") < 0) | (F.col("total_amount") < 0)).count()

In [0]:
clean = (clean
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("trip_duration_min",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60)
    .withColumn("_silver_processed_at", F.current_timestamp())
)
 
(clean.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("pickup_date")
    .saveAsTable("nyc_taxi.silver.trips_clean"))
 
(quarantine.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("nyc_taxi.silver.trips_quarantine"))


In [0]:
display(clean.limit(10))
display(quarantine.limit(10))

In [0]:
display(bronze.select("tolls_amount").orderBy(F.desc("tolls_amount")).limit(30))

In [0]:
bronze.approxQuantile("tolls_amount", [0.99, 0.999, 0.9999, 0.99999], 0.0001)

In [0]:
bronze.filter(F.col("tolls_amount") < 0).count()

In [0]:
display(bronze.filter(F.col("tolls_amount") < 0).orderBy(F.col("tolls_amount")).limit(10))

In [0]:
neg_tolls = bronze.filter(F.col("tolls_amount") < 0)
print("negative tolls, total:        ", neg_tolls.count())
print("...also negative fare:        ", neg_tolls.filter(F.col("fare_amount") < 0).count())
print("...also negative total:       ", neg_tolls.filter(F.col("total_amount") < 0).count())

In [0]:
bronze.approxQuantile("tolls_amount", [0.995, 0.999, 0.9995, 0.9998], 0.0001)

In [0]:
from pyspark.sql import functions as F

bronze = spark.table("nyc_taxi.bronze.yellow_tripdata")

DISTANCE_CEILING = 100  # 99.9th percentile was 29.79mi; next value jumped to 398,608mi (two isolated errors) — 100 sits cleanly between real and broken
TOLLS_CEILING     = 100  # no clean statistical cliff (smooth growth from $10 at p99.5 to $1,702 at max) — ceiling set by known geography instead: even a multi-bridge/tunnel route stacking every toll on the way is hard-pressed to exceed ~$60-80

bronze = bronze.withColumn("_payment_type_safe", F.coalesce(F.col("payment_type"), F.lit(-1)))

tagged = bronze.withColumn("_reject_reason",
    F.when(F.col("tpep_pickup_datetime").isNull() | F.col("tpep_dropoff_datetime").isNull(),
           "missing_timestamp")
     .when(F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime"),
           "dropoff_before_pickup")
     .when(F.col("trip_distance") <= 0,
           "zero_or_negative_distance")
     .when(F.col("trip_distance") > DISTANCE_CEILING,
           "implausible_distance")
     .when(F.col("passenger_count").isNull(),
           "null_passenger_count")
     .when(F.col("passenger_count") == 0,
           "zero_passenger_count")
     .when(F.col("passenger_count") > 6,
           "implausible_passenger_count")
     .when(F.col("tolls_amount") > TOLLS_CEILING,
           "implausible_tolls")
     .when(F.col("fare_amount") < 0,
           "negative_fare")
     .when((F.col("fare_amount") < 3.00) & (~F.col("_payment_type_safe").isin(3, 4)),
           "fare_below_minimum")
     .when(F.col("total_amount") < 0,
           "negative_total")
     .when((F.col("total_amount") < 4.50) & (~F.col("_payment_type_safe").isin(3, 4)),
           "total_below_minimum")
     .otherwise(None)
).drop("_payment_type_safe")

clean      = tagged.filter(F.col("_reject_reason").isNull()).drop("_reject_reason")
quarantine = tagged.filter(F.col("_reject_reason").isNotNull())

print("clean:     ", clean.count())
print("quarantine:", quarantine.count())
quarantine.groupBy("_reject_reason").count().orderBy(F.desc("count")).show()

In [0]:
# totals should still add up to Bronze's 41,169,720 exactly
print(clean.count() + quarantine.count())

# confirm the negative-tolls rows are indeed already being absorbed upstream,
# not slipping through untouched
clean.filter(F.col("tolls_amount") < 0).count()  # expect 0

In [0]:
clean = (clean
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("trip_duration_min",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60)
    .withColumn("_silver_processed_at", F.current_timestamp())
)
 
(clean.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("pickup_date")
    .saveAsTable("nyc_taxi.silver.trips_clean"))
 
(quarantine.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("nyc_taxi.silver.trips_quarantine"))

In [0]:
path = "/Workspace/Users/rmdd.abreu@gmail.com/nyc-taxi-fabric-lakehouse/decisions.md"

entry = """
2026-07-21 — Deepened the partitioning decision (Q11/Q12): the deciding factor between pickup_date and PULocationID isn't just current cardinality (365 vs 265) or small-files risk — it's that date is a BOUNDED partitioning key and location is UNBOUNDED. Once a calendar day passes, that partition is permanently finished; no future ingestion ever adds to it. A location partition never closes — every day, forever, adds more rows to the same 265 directories.

Consequence: the skew between busy zones (Manhattan) and quiet zones (outer boroughs) doesn't just exist once, it compounds daily, so the size gap between the largest and smallest partition keeps widening indefinitely rather than settling at a fixed ratio. This also means OPTIMIZE maintenance cost on a location-partitioned table grows without bound over time, and there is no clean way to archive old data (a location partition mixes a zone's entire history together, unlike a date partition which can simply be dropped or moved to cold storage as a whole unit once it ages out).

Rule of thumb: partition on a dimension that closes (almost always time), never on a fixed-cardinality business dimension that data keeps flowing into forever.

Resolved as: partition by pickup_date (bounded, even volume per partition), and apply ZORDER BY (PULocationID) within date partitions in Week 5 to get most of the location-filtering benefit via sharper file-level statistics, without ever making location the partition key itself.
"""

with open(path, "a") as f:
    f.write(entry)

print("done")

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
 
# A deterministic surrogate key built from the columns that identify a trip
key_cols = ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
            "PULocationID", "DOLocationID", "trip_distance", "total_amount"]
 
clean = clean.withColumn("trip_id",
    F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in key_cols]), 256))


In [0]:
%sql
DROP TABLE IF EXISTS nyc_taxi.silver.trips_clean;

In [0]:
target_name = "nyc_taxi.silver.trips_clean"
 
if not spark.catalog.tableExists(target_name):
    (clean.write.format("delta").partitionBy("pickup_date").saveAsTable(target_name))
else:
    target = DeltaTable.forName(spark, target_name)
    (target.alias("t")
        .merge(clean.alias("s"), "t.trip_id = s.trip_id")
        .whenNotMatchedInsertAll()
        .execute())
 
print("silver rows:", spark.table(target_name).count())


In [0]:
target_name = "nyc_taxi.silver.trips_clean"

if not spark.catalog.tableExists(target_name):
    (clean.write.format("delta").partitionBy("pickup_date").saveAsTable(target_name))
else:
    target = DeltaTable.forName(spark, target_name)
    (target.alias("t")
        .merge(clean.alias("s"), "t.trip_id = s.trip_id")
        .whenNotMatchedInsertAll()
        .execute())

print("silver rows:", spark.table(target_name).count())

In [0]:
(quarantine.write.format("delta")
    .mode("overwrite")
    .saveAsTable("nyc_taxi.silver.trips_quarantine"))

print("quarantine rows:", spark.table("nyc_taxi.silver.trips_quarantine").count())